# Choosing the Optimal Chunk Size for RAG — An Empirical Study

In Retrieval-Augmented Generation (RAG), we split documents into **chunks**, embed them into
a vector space, and retrieve the most relevant ones to ground an LLM's answer. The size of
those chunks is one of the most consequential — yet often under-explored — design decisions in
a RAG pipeline.

**Why does this matter so much?** Because chunk size simultaneously affects:
- **Embedding quality** — what information gets compressed into a single vector
- **Retrieval precision** — whether the right facts surface for a query
- **Generation quality** — whether the LLM receives enough (but not too much) context
- **Latency & cost** — larger chunks mean fewer vectors but longer prompts

This notebook is a hands-on, empirical investigation. We will implement multiple chunking
strategies from scratch, build a complete evaluation framework, and measure how chunk size
affects retrieval and generation across real metrics — all using local models, no API keys required.

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy matplotlib

In [ ]:
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto"
)
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)
print(f"Embedding model loaded — max sequence length: {embed_model.max_seq_length}")

In [ ]:
def generate(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        top_k=20, temperature=0.7
    )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

# Quick sanity check
print(generate("What is 2 + 2? Answer in one sentence."))

## The Fundamental Tradeoff

Chunking for RAG presents a classic **precision vs. context** tradeoff:

| | Small Chunks (e.g. 128 chars) | Large Chunks (e.g. 2048 chars) |
|---|---|---|
| **Embedding focus** | Narrow, specific meaning | Broad, averaged meaning |
| **Retrieval** | High precision — finds exact facts | High recall — captures surrounding context |
| **Context window** | Many chunks fit → diverse evidence | Few chunks fit → deep but narrow evidence |
| **Noise** | Low — each chunk is about one thing | High — irrelevant sentences dilute the signal |
| **Boundary losses** | Frequent — sentences/ideas get split | Rare — ideas stay intact |

Think of it like searching a library: small chunks are like individual index cards (easy to find
the right one, but you might miss the paragraph around it), while large chunks are like
photocopying whole pages (you get full context but wade through irrelevant text).

## Information Theory Perspective: Signal-to-Noise in Retrieval

From an information-theoretic standpoint, each chunk's embedding is a **lossy compression** of
the chunk's content into a fixed-dimensional vector (768 dims for BGE-base). The key insight:

> **Signal-to-Noise Ratio (SNR) = relevant information in chunk / total information in chunk**

- **Tiny chunks** have high SNR per-chunk but suffer from **fragmentation**: the answer may
  span multiple chunks, and no single chunk scores high enough to be retrieved.
- **Huge chunks** have low SNR because the embedding must average over many topics. The
  relevant sentence gets "diluted" by surrounding text, reducing cosine similarity with the query.

The **optimal chunk size** maximizes the probability that:
1. A single chunk contains enough relevant information to score highly against the query
2. The retrieved chunk doesn't contain so much irrelevant text that it confuses the LLM

This is fundamentally a **bias-variance tradeoff**: small chunks have high variance (fragile
retrieval), large chunks have high bias (diluted signals).

In [ ]:
# Visualize the conceptual tradeoff
import matplotlib.pyplot as plt
import numpy as np

chunk_sizes = np.array([64, 128, 256, 512, 768, 1024, 1536, 2048, 3072, 4096])

# Conceptual curves — illustrative; we'll measure real ones later
precision = 1 / (1 + np.exp(-0.005 * (chunk_sizes - 300))) * 0.3 + 0.7 * np.exp(-chunk_sizes / 1500)
recall = 1 - np.exp(-chunk_sizes / 400)
f1 = 2 * precision * recall / (precision + recall + 1e-9)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(chunk_sizes, precision, 'b-o', label='Precision (conceptual)', linewidth=2)
ax.plot(chunk_sizes, recall, 'r-s', label='Recall (conceptual)', linewidth=2)
ax.plot(chunk_sizes, f1, 'g--^', label='F1 (conceptual)', linewidth=2)
ax.axvspan(256, 1024, alpha=0.15, color='green', label='Typical "Goldilocks" zone')
ax.set_xlabel('Chunk Size (characters)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('The Chunk Size Tradeoff (Conceptual)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4200)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()
print("The green zone (256-1024) is where most RAG systems find their optimum.")

## The Goldilocks Zone: 256–1024 Tokens

Empirically, most RAG practitioners converge on chunk sizes between **256 and 1024 tokens**
(roughly 1,000–4,000 characters for English text). Here's why:

- **Below 256 tokens**: Chunks are too short to contain a complete thought. Retrieval becomes
  brittle — slight query variations miss the target chunk. The embedding model "wastes"
  capacity on partial ideas.

- **256–512 tokens**: Great for factoid-style Q&A where answers are concise. Works well with
  embedding models like BGE-base (max 512 tokens).

- **512–1024 tokens**: Better for complex questions requiring reasoning over multiple related
  facts. Good for technical/scientific text where paragraphs are information-dense.

- **Above 1024 tokens**: Usually too large for embedding models (most cap at 512 tokens,
  meaning text gets **truncated silently**). Retrieval precision drops as the embedding
  averages over too many topics.

⚠️ **Critical caveat**: These are starting points, not universal truths. The optimal size
depends on your content type, query patterns, and embedding model. That's exactly why we
need to **measure empirically** — which is what the rest of this notebook does.

---
# Section 2: Chunking Strategies — Implemented from Scratch

There are many ways to split text into chunks. Each strategy makes different assumptions about
where meaningful boundaries lie. We will implement **four strategies** from scratch and compare
them on the same text:

1. **Fixed-size character chunking** — the simplest approach, splits every *N* characters
2. **Token-aware chunking** — respects tokenizer boundaries, never splits mid-token
3. **Sentence-boundary chunking** — groups complete sentences up to a size limit
4. **Paragraph-boundary chunking** — uses natural paragraph breaks as primary dividers

In [ ]:
# Our sample text for demonstrating chunking strategies
SAMPLE_TEXT = """Machine learning is a subset of artificial intelligence that focuses on building
systems that learn from data. Unlike traditional programming where rules are explicitly coded,
machine learning algorithms discover patterns automatically.

There are three main types of machine learning. Supervised learning uses labeled training data
to learn a mapping from inputs to outputs. Common examples include classification (spam
detection, image recognition) and regression (price prediction, demand forecasting).

Unsupervised learning works with unlabeled data to discover hidden structures. Clustering
algorithms like K-means group similar data points together. Dimensionality reduction techniques
like PCA help visualize high-dimensional data.

Reinforcement learning is different from both supervised and unsupervised learning. An agent
learns by interacting with an environment, receiving rewards or penalties for its actions. This
approach has achieved remarkable results in game playing (AlphaGo) and robotics.

Deep learning, a subset of machine learning, uses neural networks with many layers. These
networks can automatically learn hierarchical representations of data. Convolutional neural
networks (CNNs) excel at image tasks, while recurrent neural networks (RNNs) and transformers
handle sequential data like text.

The transformer architecture, introduced in 2017, revolutionized natural language processing.
Models like BERT, GPT, and T5 are all based on transformers. The key innovation was the
self-attention mechanism, which allows the model to weigh the importance of different parts of
the input when producing each part of the output."""

print(f"Sample text length: {len(SAMPLE_TEXT)} characters")
print(f"Paragraphs: {len([p for p in SAMPLE_TEXT.strip().split(chr(10)+chr(10)) if p.strip()])}")

### Strategy 1: Fixed-Size Character Chunking

The simplest possible approach: split every *N* characters, regardless of words, sentences, or
paragraphs. It's fast, predictable, and deterministic — but it routinely slices through words
and sentences, creating fragments that lose meaning at chunk boundaries.

**When to use**: Prototyping, baseline comparisons, or when text has no clear structure.
**Weakness**: Almost guaranteed to split mid-sentence or even mid-word.

In [ ]:
def chunk_fixed_size(text, chunk_size, overlap=0):
    """Split text into fixed-size character chunks with optional overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]

# Demo
chunks = chunk_fixed_size(SAMPLE_TEXT, chunk_size=200)
print(f"Fixed-size chunking (200 chars, no overlap): {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:80] + "..." if len(chunk) > 80 else chunk)
    print()

### Strategy 2: Token-Aware Chunking

Instead of counting characters, we count **tokens** — the units the model actually processes.
This ensures we never split in the middle of a multi-byte or multi-character token, and gives
us precise control over how much each chunk consumes in the model's context window.

**When to use**: When you need exact control over token budget per chunk (e.g., fitting a
specific number of chunks into a fixed-size context window).

In [ ]:
def chunk_by_tokens(text, tokenizer, max_tokens, overlap_tokens=0):
    """Split text into chunks of at most max_tokens tokens."""
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)
        start += max_tokens - overlap_tokens
    return chunks

# Demo
chunks_tok = chunk_by_tokens(SAMPLE_TEXT, tokenizer, max_tokens=50)
print(f"Token-aware chunking (50 tokens, no overlap): {len(chunks_tok)} chunks\n")
for i, chunk in enumerate(chunks_tok[:5]):
    n_tokens = len(tokenizer.encode(chunk))
    print(f"--- Chunk {i+1} ({n_tokens} tokens, {len(chunk)} chars) ---")
    print(chunk[:100] + "..." if len(chunk) > 100 else chunk)
    print()
print(f"... and {len(chunks_tok) - 5} more chunks")

### Strategy 3: Sentence-Boundary Chunking

This strategy uses **sentence-ending punctuation** as the unit of division, packing as many
complete sentences as possible into each chunk without exceeding a character limit. No sentence
is ever split, so each chunk reads naturally.

**When to use**: General-purpose text retrieval. Sentences are the smallest self-contained
semantic units in prose, making this an excellent default strategy.

In [ ]:
import re

def chunk_by_sentences(text, max_chunk_size=500):
    """Split text into chunks at sentence boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if current_chunk and len(current_chunk) + len(sentence) + 1 > max_chunk_size:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk = (current_chunk + " " + sentence).strip()

    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# Demo
chunks_sent = chunk_by_sentences(SAMPLE_TEXT, max_chunk_size=400)
print(f"Sentence-boundary chunking (max 400 chars): {len(chunks_sent)} chunks\n")
for i, chunk in enumerate(chunks_sent):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:120] + "..." if len(chunk) > 120 else chunk)
    print()

### Strategy 4: Paragraph-Boundary Chunking

Paragraphs are the author's own grouping of related ideas. Splitting at paragraph boundaries
respects the document's **semantic structure** — each chunk is likely about one coherent topic.
Small paragraphs are merged to avoid wastefully tiny chunks.

**When to use**: Well-structured text with clear paragraph breaks (articles, documentation,
textbooks). This is often the best strategy for educational and technical content.

In [ ]:
def chunk_by_paragraphs(text, max_chunk_size=800, min_chunk_size=100):
    """Split text at paragraph boundaries, merging small paragraphs."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if current_chunk and len(current_chunk) + len(para) + 2 > max_chunk_size:
            chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            current_chunk = (current_chunk + "\n\n" + para).strip()

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    # Merge any trailing tiny chunks into the previous one
    merged = []
    for chunk in chunks:
        if merged and len(chunk) < min_chunk_size:
            merged[-1] = merged[-1] + "\n\n" + chunk
        else:
            merged.append(chunk)
    return merged

# Demo
chunks_para = chunk_by_paragraphs(SAMPLE_TEXT, max_chunk_size=600)
print(f"Paragraph-boundary chunking (max 600 chars): {len(chunks_para)} chunks\n")
for i, chunk in enumerate(chunks_para):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:120] + "..." if len(chunk) > 120 else chunk)
    print()

### Visual Comparison: Where Do the Boundaries Fall?

The plot below visualizes how each strategy carves the same text into chunks. Each numbered
colored block represents one chunk; its width is proportional to the chunk's character length.
Notice how paragraph-boundary chunking produces the fewest, most semantically coherent chunks,
while fixed-size chunking creates the most uniform but semantically arbitrary divisions.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 6), sharex=True)
strategies = [
    ("Fixed-size (200 chars)", chunk_fixed_size(SAMPLE_TEXT, 200)),
    ("Token-aware (50 tokens)", chunk_by_tokens(SAMPLE_TEXT, tokenizer, 50)),
    ("Sentence-boundary (400 chars)", chunk_by_sentences(SAMPLE_TEXT, 400)),
    ("Paragraph-boundary (600 chars)", chunk_by_paragraphs(SAMPLE_TEXT, 600)),
]

colors = plt.cm.Set3(np.linspace(0, 1, 12))
text_len = len(SAMPLE_TEXT)

for ax, (name, chunks) in zip(axes, strategies):
    offset = 0
    for i, chunk in enumerate(chunks):
        width = len(chunk)
        ax.barh(0, width, left=offset, height=0.6,
                color=colors[i % len(colors)], edgecolor='black', linewidth=0.5)
        if width > text_len * 0.04:
            ax.text(offset + width/2, 0, str(i+1), ha='center', va='center', fontsize=8)
        offset += width
    ax.set_xlim(0, text_len)
    ax.set_yticks([])
    ax.set_ylabel(name, fontsize=9, rotation=0, ha='right', va='center')

axes[-1].set_xlabel('Character position in document', fontsize=11)
fig.suptitle('Chunk Boundaries: Four Strategies on the Same Text', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Each numbered colored block is one chunk. Notice how paragraph-boundary\n"
      "chunking produces the fewest, most semantically coherent chunks.")

---
# Section 3: Overlap Strategies — Preventing Boundary Information Loss

When we cut text into chunks, we inevitably create **boundaries** where sentences or ideas get
split. Overlap is the primary mitigation: by including some text from the end of chunk *N* at
the beginning of chunk *N+1*, we ensure that information near boundaries appears in at least
two chunks.

**Key questions we'll answer:**
- How much overlap is enough?
- When does overlap stop helping and start wasting storage/compute?
- How does overlap interact with chunk size?

In [ ]:
def chunk_with_overlap(text, chunk_size, overlap, mode="chars"):
    """
    Chunk text with configurable overlap.
    mode='chars' -> character-based; mode='sentences' -> sentence-based with char limit.
    """
    if mode == "chars":
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunks.append(text[start:end].strip())
            step = chunk_size - overlap
            if step <= 0:
                step = 1  # safety: avoid infinite loop
            start += step
        return [c for c in chunks if c]

    elif mode == "sentences":
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        chunks = []
        current = []
        current_len = 0

        for sent in sentences:
            if current_len + len(sent) > chunk_size and current:
                chunks.append(" ".join(current))
                # Keep last N chars worth of sentences as overlap
                overlap_text = ""
                overlap_sents = []
                for s in reversed(current):
                    if len(overlap_text) + len(s) <= overlap:
                        overlap_sents.insert(0, s)
                        overlap_text = " ".join(overlap_sents)
                    else:
                        break
                current = overlap_sents
                current_len = len(overlap_text)
            current.append(sent)
            current_len += len(sent) + 1

        if current:
            chunks.append(" ".join(current))
        return chunks

# Demo: same text, same chunk size, varying overlap
print("Effect of overlap on chunk count and boundary coverage:\n")
for overlap_pct in [0, 10, 20, 30, 50]:
    overlap = int(300 * overlap_pct / 100)
    chunks = chunk_with_overlap(SAMPLE_TEXT, chunk_size=300, overlap=overlap)
    print(f"  Overlap {overlap_pct:2d}% ({overlap:3d} chars): {len(chunks):2d} chunks, "
          f"total chars stored = {sum(len(c) for c in chunks)}")

### The Cost of Overlap: Storage vs. Boundary Recovery

Overlap has a direct cost: **more chunks to store and embed**. If you use 50% overlap, you
roughly double your index size. The question is whether the boundary information you recover is
worth that cost. The next cell measures this tradeoff quantitatively.

In [ ]:
# Visualize: How overlap affects chunk count and storage

overlaps = list(range(0, 201, 10))
chunk_counts = []
storage_ratios = []

for ov in overlaps:
    chunks = chunk_with_overlap(SAMPLE_TEXT, 300, ov)
    chunk_counts.append(len(chunks))
    total_stored = sum(len(c) for c in chunks)
    storage_ratios.append(total_stored / len(SAMPLE_TEXT))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(overlaps, chunk_counts, 'b-o', markersize=4)
ax1.set_xlabel('Overlap (characters)', fontsize=11)
ax1.set_ylabel('Number of Chunks', fontsize=11, color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1_twin = ax1.twinx()
ax1_twin.plot(overlaps, storage_ratios, 'r-s', markersize=4)
ax1_twin.set_ylabel('Storage Ratio (total / original)', fontsize=11, color='red')
ax1_twin.tick_params(axis='y', labelcolor='red')
ax1.set_title('Overlap → More Chunks & Higher Storage', fontsize=12)
ax1.grid(True, alpha=0.3)

# Overlap as % of chunk size vs storage ratio
pct_labels = [f"{int(ov/300*100)}%" for ov in overlaps[::4]]
ax2.bar(range(len(overlaps)), storage_ratios, width=0.8, color='coral', alpha=0.7)
ax2.set_xlabel('Overlap (characters)', fontsize=11)
ax2.set_ylabel('Storage Ratio', fontsize=11)
ax2.set_title('Storage Cost Grows Linearly with Overlap', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print("Key insight: overlap has diminishing returns beyond ~20% of chunk size.\n"
      "Storage grows linearly, but boundary recovery plateaus quickly.")

### The Diminishing Returns of Excessive Overlap

The plots above reveal an important pattern:

1. **Storage cost grows linearly** with overlap — 50% overlap means storing ~2× the original text
2. **Boundary recovery plateaus** — once overlap exceeds a typical sentence length (~100-150 chars),
   adding more overlap doesn't rescue additional split sentences
3. **Recommended overlap**: **10–20%** of chunk size is the sweet spot for most applications

**Rule of thumb**: If your chunk size is 500 characters, use 50–100 characters of overlap.
This ensures most sentence boundaries are covered without ballooning your index size.

In [ ]:
# Practical demo: see exactly how overlap changes content at boundaries
chunks_no_overlap = chunk_with_overlap(SAMPLE_TEXT, 300, 0)
chunks_with_overlap = chunk_with_overlap(SAMPLE_TEXT, 300, 60)

print("=== WITHOUT overlap: Chunk 2 → 3 boundary ===")
print(f"End of chunk 2:   ...{chunks_no_overlap[1][-80:]}")
print(f"Start of chunk 3:  {chunks_no_overlap[2][:80]}...")
print()
print("=== WITH 60-char overlap: Chunk 2 → 3 boundary ===")
print(f"End of chunk 2:   ...{chunks_with_overlap[1][-80:]}")
print(f"Start of chunk 3:  {chunks_with_overlap[2][:80]}...")
print()
print("Notice how the overlapping version ensures the sentence at the boundary")
print("appears completely in at least one chunk.")

---
# Section 4: Empirical Evaluation — Measuring What Matters

Theory is useful, but RAG is fundamentally empirical. In this section we build a rigorous
evaluation pipeline:

1. **Create** a synthetic knowledge base with **known ground truth** (12 topics)
2. **Define** test queries where we know exactly which chunks should be retrieved
3. **Chunk** the same text at 5 different sizes (128, 256, 512, 1024, 2048 chars)
4. **Embed** and **index** each chunk set with FAISS
5. **Measure** Precision@3, Recall@3, and average cosine similarity
6. **Generate** RAG answers at each chunk size and measure quality
7. **Plot** everything to find the empirical optimum

In [ ]:
# Build a synthetic knowledge base — 12 paragraphs covering different topics
# Each paragraph is labeled so we know which ones are relevant to each query

KNOWLEDGE_BASE = {
    "solar_system": (
        "The Solar System consists of the Sun and the objects that orbit it, including "
        "eight planets, their moons, dwarf planets, asteroids, and comets. The four inner "
        "planets — Mercury, Venus, Earth, and Mars — are rocky terrestrial worlds. The four "
        "outer planets — Jupiter, Saturn, Uranus, and Neptune — are gas and ice giants. "
        "Jupiter is the largest planet, with a mass more than twice that of all other planets "
        "combined. The asteroid belt between Mars and Jupiter contains millions of rocky objects."
    ),
    "photosynthesis": (
        "Photosynthesis is the process by which green plants, algae, and some bacteria convert "
        "light energy into chemical energy stored in glucose. The process occurs in chloroplasts, "
        "specifically in structures called thylakoids. The light-dependent reactions capture solar "
        "energy and produce ATP and NADPH. The Calvin cycle (light-independent reactions) uses "
        "these energy carriers to fix carbon dioxide into glucose. A single leaf can fix about "
        "5 micromoles of CO2 per square meter per second under optimal conditions."
    ),
    "neural_networks": (
        "Artificial neural networks are computing systems inspired by biological neural networks "
        "in the brain. They consist of layers of interconnected nodes (neurons) that process "
        "information. Each connection has a weight that is adjusted during training. The "
        "backpropagation algorithm, combined with gradient descent, is the standard method for "
        "training neural networks. Deep networks with many layers can learn hierarchical "
        "representations, enabling breakthroughs in image recognition, speech processing, and "
        "natural language understanding."
    ),
    "quantum_computing": (
        "Quantum computing harnesses quantum mechanical phenomena — superposition and entanglement "
        "— to process information in fundamentally different ways than classical computers. A "
        "quantum bit (qubit) can exist in a superposition of 0 and 1 simultaneously, unlike a "
        "classical bit. Quantum computers excel at certain problems like integer factorization "
        "(Shor's algorithm), database search (Grover's algorithm), and simulating quantum systems. "
        "Current quantum computers have 50-1000+ qubits but are limited by noise and decoherence."
    ),
    "roman_empire": (
        "The Roman Empire was one of the largest and most influential civilizations in history, "
        "lasting from 27 BC to 476 AD in the West. At its peak under Emperor Trajan around 117 AD, "
        "it controlled the entire Mediterranean basin, much of Western Europe, and parts of the "
        "Middle East. Roman engineering achievements include aqueducts, roads, and concrete "
        "construction. The empire's legal system formed the basis of civil law in many modern nations. "
        "Latin, the empire's language, evolved into the Romance languages."
    ),
    "climate_change": (
        "Climate change refers to long-term shifts in global temperatures and weather patterns. "
        "Since the Industrial Revolution, human activities — primarily burning fossil fuels — have "
        "increased atmospheric CO2 from 280 ppm to over 420 ppm. This enhanced greenhouse effect "
        "has raised global average temperatures by approximately 1.1 degrees Celsius since pre-industrial times. "
        "Consequences include rising sea levels (3.7 mm/year), more frequent extreme weather events, "
        "ocean acidification, and biodiversity loss across ecosystems."
    ),
    "dna_structure": (
        "DNA (deoxyribonucleic acid) is the molecule that carries genetic information in all living "
        "organisms. Its double-helix structure was discovered by Watson and Crick in 1953, building "
        "on X-ray crystallography work by Rosalind Franklin. DNA consists of two complementary "
        "strands of nucleotides, each containing a sugar (deoxyribose), a phosphate group, and one "
        "of four bases: adenine (A), thymine (T), guanine (G), and cytosine (C). Base pairing rules "
        "(A-T, G-C) enable accurate replication during cell division."
    ),
    "black_holes": (
        "Black holes are regions of spacetime where gravity is so strong that nothing, not even "
        "light, can escape. They form when massive stars collapse at the end of their life cycle. "
        "The boundary of no return is called the event horizon. Stellar black holes typically have "
        "masses of 5-50 solar masses, while supermassive black holes at galaxy centers can have "
        "billions of solar masses. In 2019, the Event Horizon Telescope captured the first direct "
        "image of a black hole's shadow in galaxy M87."
    ),
    "ml_training": (
        "Training a machine learning model involves optimizing its parameters to minimize a loss "
        "function. The most common optimization algorithm is stochastic gradient descent (SGD), "
        "which updates parameters using gradients computed on mini-batches of data. Learning rate "
        "is a critical hyperparameter: too high causes divergence, too low causes slow convergence. "
        "Modern optimizers like Adam combine momentum and adaptive learning rates. Regularization "
        "techniques (dropout, weight decay, early stopping) prevent overfitting to training data."
    ),
    "ocean_currents": (
        "Ocean currents are continuous movements of seawater driven by wind, temperature differences, "
        "salinity gradients, and the Earth's rotation (Coriolis effect). The Gulf Stream carries warm "
        "water from the Gulf of Mexico northeastward across the Atlantic, moderating European "
        "climates. The global thermohaline circulation — sometimes called the ocean conveyor belt — "
        "moves water between the surface and deep ocean over centuries. Changes in ocean currents "
        "can significantly affect regional climates and marine ecosystems."
    ),
    "renaissance_art": (
        "The Renaissance was a cultural movement that began in Italy in the 14th century and spread "
        "across Europe. It marked a revival of interest in classical Greek and Roman art, philosophy, "
        "and science. Key innovations in art included linear perspective (Brunelleschi), chiaroscuro "
        "(light-shadow modeling), and anatomical accuracy. Leonardo da Vinci, Michelangelo, and "
        "Raphael are considered the three great masters. The period produced masterworks like the "
        "Mona Lisa, the Sistine Chapel ceiling, and The School of Athens."
    ),
    "immune_system": (
        "The human immune system is a complex network of cells, tissues, and organs that defends "
        "the body against pathogens. Innate immunity provides immediate, non-specific defense "
        "through physical barriers (skin, mucous membranes), phagocytes, and inflammation. Adaptive "
        "immunity develops specific responses through T cells and B cells, which can recognize and "
        "remember specific pathogens. Vaccines work by training the adaptive immune system to "
        "recognize pathogens without causing disease. Immunological memory enables faster, stronger "
        "responses upon re-exposure."
    ),
}

FULL_TEXT = "\n\n".join(KNOWLEDGE_BASE.values())
print(f"Knowledge base: {len(KNOWLEDGE_BASE)} topics, {len(FULL_TEXT)} characters total")
print(f"Average paragraph length: {len(FULL_TEXT) // len(KNOWLEDGE_BASE)} chars")
print(f"\nTopics: {', '.join(KNOWLEDGE_BASE.keys())}")

In [ ]:
# Define test queries with ground-truth relevant topics
TEST_QUERIES = [
    {
        "query": "What is the largest planet in the Solar System?",
        "relevant": ["solar_system"],
        "expected_answer": "Jupiter"
    },
    {
        "query": "How does photosynthesis convert light into energy?",
        "relevant": ["photosynthesis"],
        "expected_answer": "chloroplasts, ATP, Calvin cycle"
    },
    {
        "query": "What algorithm is used to train neural networks?",
        "relevant": ["neural_networks", "ml_training"],
        "expected_answer": "backpropagation, gradient descent"
    },
    {
        "query": "How many qubits do current quantum computers have?",
        "relevant": ["quantum_computing"],
        "expected_answer": "50-1000+"
    },
    {
        "query": "When did the Roman Empire reach its peak territory?",
        "relevant": ["roman_empire"],
        "expected_answer": "117 AD under Trajan"
    },
    {
        "query": "How much has global CO2 increased since pre-industrial times?",
        "relevant": ["climate_change"],
        "expected_answer": "280 to over 420 ppm"
    },
    {
        "query": "Who discovered the structure of DNA?",
        "relevant": ["dna_structure"],
        "expected_answer": "Watson and Crick"
    },
    {
        "query": "What was the first black hole ever photographed?",
        "relevant": ["black_holes"],
        "expected_answer": "M87"
    },
    {
        "query": "What is the role of the Gulf Stream in climate?",
        "relevant": ["ocean_currents"],
        "expected_answer": "moderating European climates"
    },
    {
        "query": "How do vaccines work with the immune system?",
        "relevant": ["immune_system"],
        "expected_answer": "training adaptive immune system"
    },
]

print(f"Defined {len(TEST_QUERIES)} test queries with ground-truth labels\n")
for i, q in enumerate(TEST_QUERIES):
    print(f"  Q{i+1}: {q['query']}")
    print(f"       Relevant: [{', '.join(q['relevant'])}]  |  Expected: {q['expected_answer']}\n")

### Building the Ground-Truth Labeling System

To measure precision and recall, we need to know which chunks are "relevant" to each query.
Since we built the knowledge base ourselves, we can map every character position to its source
topic. When a chunk overlaps with the topic paragraph that answers a query, we label it as
relevant. This gives us a **perfect ground truth** for evaluation.

In [ ]:
import faiss
import time

def build_position_to_topic_map(knowledge_base):
    """Map each character position in FULL_TEXT to its topic name."""
    pos_map = {}
    offset = 0
    for topic, text in knowledge_base.items():
        for i in range(len(text)):
            pos_map[offset + i] = topic
        offset += len(text) + 2  # +2 for "\n\n" separator
    return pos_map

def label_chunk(chunk, pos_map, full_text):
    """Determine which topics a chunk covers (by character overlap)."""
    start = full_text.find(chunk[:50])
    if start == -1:
        start = 0
    topic_counts = {}
    for i in range(start, min(start + len(chunk), len(full_text))):
        topic = pos_map.get(i, "unknown")
        topic_counts[topic] = topic_counts.get(topic, 0) + 1
    # Return topics that cover at least 20% of the chunk
    threshold = len(chunk) * 0.2
    return [t for t, c in topic_counts.items() if c >= threshold]

pos_map = build_position_to_topic_map(KNOWLEDGE_BASE)
print(f"Position map built: {len(pos_map)} character positions mapped to {len(KNOWLEDGE_BASE)} topics")

# Verify labeling
test_chunk = list(KNOWLEDGE_BASE.values())[0][:100]
labels = label_chunk(test_chunk, pos_map, FULL_TEXT)
print(f"Verification: first 100 chars of 'solar_system' → labeled as: {labels} ✓")

In [ ]:
CHUNK_SIZES = [128, 256, 512, 1024, 2048]
TOP_K = 3

def evaluate_chunk_size(text, chunk_size, queries, pos_map, top_k=3):
    """Full evaluation pipeline for a given chunk size."""
    # 1. Chunk the text with 15% overlap
    overlap = int(chunk_size * 0.15)
    chunks = chunk_with_overlap(text, chunk_size, overlap, mode="chars")

    # 2. Label each chunk with its ground-truth topics
    chunk_labels = [label_chunk(c, pos_map, text) for c in chunks]

    # 3. Embed all chunks
    t0 = time.time()
    embeddings = embed_model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
    embed_time = time.time() - t0

    # 4. Build FAISS index (inner product = cosine sim for normalized vectors)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))

    # 5. Evaluate each query
    precisions, recalls, relevance_scores, retrieval_times = [], [], [], []

    for q in queries:
        q_embed = embed_model.encode([q["query"]], normalize_embeddings=True)

        t0 = time.time()
        scores, indices = index.search(q_embed.astype(np.float32), top_k)
        retrieval_times.append(time.time() - t0)

        retrieved_topics = set()
        relevant_count = 0
        for idx in indices[0]:
            if idx < len(chunk_labels):
                topics = chunk_labels[idx]
                retrieved_topics.update(topics)
                if any(t in q["relevant"] for t in topics):
                    relevant_count += 1

        precisions.append(relevant_count / top_k)
        recalls.append(len(set(q["relevant"]) & retrieved_topics) / len(q["relevant"]))
        relevance_scores.append(float(np.mean(scores[0])))

    return {
        "chunk_size": chunk_size,
        "n_chunks": len(chunks),
        "avg_chunk_len": np.mean([len(c) for c in chunks]),
        "embed_time": embed_time,
        "precision@3": np.mean(precisions),
        "recall@3": np.mean(recalls),
        "avg_relevance_score": np.mean(relevance_scores),
        "avg_retrieval_time": np.mean(retrieval_times),
        "chunks": chunks,
        "index": index,
        "chunk_labels": chunk_labels,
    }

# Run evaluation for all chunk sizes
results = {}
print(f"{'Size':>6} | {'Chunks':>6} | {'P@3':>6} | {'R@3':>6} | {'Rel Score':>9} | {'Embed(s)':>8} | {'Retr(ms)':>8}")
print("-" * 72)

for cs in CHUNK_SIZES:
    r = evaluate_chunk_size(FULL_TEXT, cs, TEST_QUERIES, pos_map, TOP_K)
    results[cs] = r
    print(f"{cs:>6} | {r['n_chunks']:>6} | {r['precision@3']:>6.3f} | {r['recall@3']:>6.3f} | "
          f"{r['avg_relevance_score']:>9.4f} | {r['embed_time']:>8.3f} | {r['avg_retrieval_time']*1000:>8.3f}")

print("\nEvaluation complete for all chunk sizes.")

### Interpreting the Retrieval Metrics

- **Precision@3**: Of the 3 retrieved chunks, what fraction contained the correct topic?
  Higher precision = less noise in the LLM's context.
- **Recall@3**: Did we retrieve at least one chunk from every relevant topic?
  Higher recall = we didn't miss any important information.
- **Relevance Score**: Average cosine similarity between queries and their top-3 chunks.
  Higher = the embedding space thinks these chunks are a good match.

Let's visualize these metrics to see the patterns clearly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sizes = [r["chunk_size"] for r in results.values()]
p3 = [r["precision@3"] for r in results.values()]
r3 = [r["recall@3"] for r in results.values()]
rel = [r["avg_relevance_score"] for r in results.values()]

# Plot 1: Precision and Recall
ax = axes[0]
ax.plot(sizes, p3, 'b-o', label='Precision@3', linewidth=2, markersize=8)
ax.plot(sizes, r3, 'r-s', label='Recall@3', linewidth=2, markersize=8)
f1 = [2*p*r/(p+r) if (p+r) > 0 else 0 for p, r in zip(p3, r3)]
ax.plot(sizes, f1, 'g--^', label='F1@3', linewidth=2, markersize=8)
ax.set_xlabel('Chunk Size (characters)', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Retrieval Quality vs Chunk Size', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
ax.set_xscale('log', base=2)
ax.set_xticks(sizes)
ax.set_xticklabels(sizes)

# Plot 2: Average cosine similarity
ax = axes[1]
ax.bar(range(len(sizes)), rel, color='teal', alpha=0.7, width=0.6)
ax.set_xticks(range(len(sizes)))
ax.set_xticklabels(sizes)
ax.set_xlabel('Chunk Size (characters)', fontsize=11)
ax.set_ylabel('Avg Cosine Similarity', fontsize=11)
ax.set_title('Retrieval Relevance Score', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Number of chunks
ax = axes[2]
n_chunks = [r["n_chunks"] for r in results.values()]
ax.bar(range(len(sizes)), n_chunks, color='coral', alpha=0.7, width=0.6)
ax.set_xticks(range(len(sizes)))
ax.set_xticklabels(sizes)
ax.set_xlabel('Chunk Size (characters)', fontsize=11)
ax.set_ylabel('Number of Chunks', fontsize=11)
ax.set_title('Index Size (Chunk Count)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print("Left: Precision tends to peak at moderate sizes; recall improves with larger chunks.")
print("Center: Relevance scores show how well queries match their best chunks.")
print("Right: Smaller chunks mean more vectors to store and search.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

embed_times = [r["embed_time"] for r in results.values()]
retr_times = [r["avg_retrieval_time"] * 1000 for r in results.values()]

ax1.bar(range(len(sizes)), embed_times, color='steelblue', alpha=0.7, width=0.6)
ax1.set_xticks(range(len(sizes)))
ax1.set_xticklabels(sizes)
ax1.set_xlabel('Chunk Size (characters)', fontsize=11)
ax1.set_ylabel('Embedding Time (seconds)', fontsize=11)
ax1.set_title('Time to Embed All Chunks', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')

ax2.bar(range(len(sizes)), retr_times, color='darkorange', alpha=0.7, width=0.6)
ax2.set_xticks(range(len(sizes)))
ax2.set_xticklabels(sizes)
ax2.set_xlabel('Chunk Size (characters)', fontsize=11)
ax2.set_ylabel('Avg Retrieval Time (ms)', fontsize=11)
ax2.set_title('FAISS Search Latency per Query', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print("Smaller chunks = more vectors = slightly longer embed time but FAISS search")
print("stays fast regardless. In production with millions of chunks, index type matters more.")

### End-to-End RAG Quality: Does Retrieval Translate to Better Answers?

Good retrieval metrics don't automatically guarantee good generation. A chunk might be
"relevant" by topic overlap but still lack the specific sentence the LLM needs, or a large
chunk might contain the answer buried in noise that confuses the model.

Let's perform full RAG — retrieve top-3 chunks and have the LLM generate answers — at three
representative chunk sizes, then measure how often the generated answer contains the expected
keywords.

In [ ]:
def rag_generate(query, chunks, index, top_k=3):
    """Perform RAG: retrieve relevant chunks and generate an answer."""
    q_embed = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(q_embed.astype(np.float32), top_k)

    context_chunks = [chunks[i] for i in indices[0] if i < len(chunks)]
    context = "\n---\n".join(context_chunks)

    prompt = f"""Answer the question based ONLY on the provided context.
Be concise and specific. If the context doesn't contain the answer, say "Not found in context."

Context:
{context}

Question: {query}
Answer:"""

    return generate(prompt, max_new_tokens=150)

# Test on a subset of queries (LLM generation is slow)
print("RAG Generation Quality at Different Chunk Sizes\n" + "=" * 55)

eval_queries = TEST_QUERIES[:4]

for cs in [256, 512, 1024]:
    r = results[cs]
    print(f"\n{'='*55}")
    print(f"CHUNK SIZE: {cs} characters ({r['n_chunks']} chunks)")
    print(f"{'='*55}")
    for q in eval_queries:
        answer = rag_generate(q["query"], r["chunks"], r["index"])
        contains_expected = any(
            exp.lower() in answer.lower()
            for exp in q["expected_answer"].split(", ")
        )
        status = "✓" if contains_expected else "✗"
        print(f"\n  Q: {q['query']}")
        print(f"  A: {answer[:200]}")
        print(f"  Expected keywords: {q['expected_answer']} → {status}")

In [ ]:
# Score generation quality: what fraction of expected keywords appear in answers?
print("Generation Quality Score by Chunk Size\n" + "-" * 45)

gen_scores = {}
for cs in [256, 512, 1024]:
    r = results[cs]
    correct = 0
    total = 0
    for q in eval_queries:
        answer = rag_generate(q["query"], r["chunks"], r["index"])
        keywords = q["expected_answer"].split(", ")
        hits = sum(1 for kw in keywords if kw.lower() in answer.lower())
        correct += hits
        total += len(keywords)
    gen_scores[cs] = correct / total if total > 0 else 0
    print(f"  Chunk size {cs:>5}: {gen_scores[cs]:.1%} of expected keywords found")

# Plot generation quality alongside retrieval metrics
fig, ax = plt.subplots(figsize=(8, 5))
cs_list = sorted(gen_scores.keys())
gen_vals = [gen_scores[cs] for cs in cs_list]
p3_subset = [results[cs]["precision@3"] for cs in cs_list]
r3_subset = [results[cs]["recall@3"] for cs in cs_list]

x = np.arange(len(cs_list))
width = 0.25
ax.bar(x - width, p3_subset, width, label='Precision@3', color='steelblue', alpha=0.8)
ax.bar(x, r3_subset, width, label='Recall@3', color='coral', alpha=0.8)
ax.bar(x + width, gen_vals, width, label='Generation Quality', color='seagreen', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(cs_list)
ax.set_xlabel('Chunk Size (characters)', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Retrieval vs Generation Quality', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()
print("Generation quality depends on BOTH good retrieval AND appropriate context length.")
print("Even with perfect retrieval, too-large chunks may confuse the LLM with noise.")

---
# Section 5: Advanced Considerations

## Content-Type Specific Chunking

Different content types have fundamentally different structures. Chunking code the same way
you chunk prose is a recipe for poor retrieval:

| Content Type | Natural Boundaries | Recommended Strategy |
|---|---|---|
| **Prose / articles** | Paragraphs, sections | Sentence or paragraph-based, 256–512 tokens |
| **Source code** | Functions, classes, blocks | AST-aware or function-level chunking |
| **Tables / CSV** | Rows, columns | Row-group chunking with header repetition |
| **Legal documents** | Clauses, sections, articles | Section-boundary with numbered hierarchy |
| **Chat logs** | Conversation turns | Turn-based with sliding window |

The key insight: **match your chunk boundaries to the natural semantic units** of your content.

In [ ]:
# Demo: How the same chunking strategy produces very different results on code vs prose

CODE_SAMPLE = """def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result"""

PROSE_SAMPLE = (
    "Binary search is an efficient algorithm for finding an item in a sorted list. "
    "It works by repeatedly dividing the search interval in half. If the target value "
    "is less than the middle element, the search continues in the lower half. Otherwise, "
    "it continues in the upper half. The time complexity is O(log n), making it much "
    "faster than linear search for large datasets. Merge sort is a divide-and-conquer "
    "sorting algorithm. It divides the input array into two halves, recursively sorts "
    "each half, and then merges the sorted halves. Its time complexity is O(n log n) "
    "in all cases, making it more predictable than quicksort."
)

# Chunk both at 200 chars with fixed-size strategy
code_chunks = chunk_fixed_size(CODE_SAMPLE, 200)
prose_chunks = chunk_fixed_size(PROSE_SAMPLE, 200)

print("=== CODE chunked at 200 chars (fixed-size) ===")
for i, c in enumerate(code_chunks):
    # Check if a function definition got split
    has_def = "def " in c
    ends_clean = c.rstrip().endswith(":")  or c.rstrip().endswith("return -1")
    print(f"\nChunk {i+1} ({len(c)} chars):")
    print(c[:150])
    if not ends_clean and has_def:
        print("  ⚠️  Function body split across chunks!")

print("\n\n=== PROSE chunked at 200 chars (fixed-size) ===")
for i, c in enumerate(prose_chunks):
    print(f"\nChunk {i+1} ({len(c)} chars):")
    print(c[:150])

print("\n💡 For code, a function-aware chunker would keep each function intact,")
print("   dramatically improving retrieval for queries like 'how does merge sort work?'")

## Dynamic Chunk Sizing Based on Semantic Density

Not all text is equally information-dense. A paragraph of dense technical content may pack
more retrievable facts per character than a paragraph of narrative filler. **Semantic density
analysis** measures how much unique information each chunk contributes:

- **High consecutive similarity** between adjacent chunks → redundancy (chunks are too similar)
- **High centroid similarity** (all chunks similar to the average) → poor discriminability
- **Ideal**: moderate consecutive similarity, low centroid similarity = diverse, distinct chunks

In [ ]:
def analyze_semantic_density(text, chunk_sizes):
    """Measure embedding variance as a proxy for semantic density."""
    results_density = {}
    for cs in chunk_sizes:
        chunks = chunk_with_overlap(text, cs, int(cs * 0.1))
        embeddings = embed_model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)

        if len(embeddings) > 1:
            consecutive_sims = [
                float(np.dot(embeddings[i], embeddings[i + 1]))
                for i in range(len(embeddings) - 1)
            ]
            avg_consecutive_sim = np.mean(consecutive_sims)
        else:
            avg_consecutive_sim = 1.0

        centroid = np.mean(embeddings, axis=0)
        distances = [float(np.dot(e, centroid)) for e in embeddings]

        results_density[cs] = {
            "n_chunks": len(chunks),
            "avg_consecutive_sim": avg_consecutive_sim,
            "avg_centroid_sim": np.mean(distances),
            "embedding_std": float(np.std(embeddings)),
        }
    return results_density

density = analyze_semantic_density(FULL_TEXT, CHUNK_SIZES)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

cs_list = sorted(density.keys())
consec = [density[cs]["avg_consecutive_sim"] for cs in cs_list]
centroid = [density[cs]["avg_centroid_sim"] for cs in cs_list]

ax1.plot(cs_list, consec, 'b-o', linewidth=2, markersize=8, label='Consecutive chunk similarity')
ax1.plot(cs_list, centroid, 'r-s', linewidth=2, markersize=8, label='Chunk-to-centroid similarity')
ax1.set_xlabel('Chunk Size (characters)', fontsize=11)
ax1.set_ylabel('Cosine Similarity', fontsize=11)
ax1.set_title('Semantic Coherence vs Chunk Size', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

stds = [density[cs]["embedding_std"] for cs in cs_list]
ax2.bar(range(len(cs_list)), stds, color='purple', alpha=0.6, width=0.6)
ax2.set_xticks(range(len(cs_list)))
ax2.set_xticklabels(cs_list)
ax2.set_xlabel('Chunk Size (characters)', fontsize=11)
ax2.set_ylabel('Embedding Std Dev', fontsize=11)
ax2.set_title('Embedding Diversity (Higher = More Distinct Chunks)', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print("When consecutive similarity is high, adjacent chunks are semantically similar → redundancy.")
print("When centroid similarity is high, all chunks look alike → poor discriminability.")
print("We want moderate consecutive similarity and LOW centroid similarity (diverse chunks).")

## Chunk Size vs. Embedding Model Max Tokens

A critical but often overlooked issue: **embedding models have a maximum input length**.
BGE-base-en-v1.5 has a maximum of **512 tokens** (~2000 characters for English). Text beyond
this limit is **silently truncated** — the embedding only represents the first 512 tokens.

This means:
- A 2048-character chunk likely fits within 512 tokens ✓
- A 4096-character chunk will be truncated — the second half is **invisible** to retrieval
- If the answer happens to be in the truncated portion, retrieval will fail silently

**This is the single most common mistake** in RAG chunk sizing: using chunks larger than the
embedding model can handle.

In [ ]:
# Demonstrate the truncation problem empirically

long_text = (
    "The first part of this text discusses the history of computing, starting with "
    "Charles Babbage's Analytical Engine in the 1830s. " * 10 +
    "THE SECRET ANSWER IS 42. This critical fact appears only in the second half. " +
    "Additional filler text to push length beyond embedding model limits. " * 10
)

print(f"Long text: {len(long_text)} characters")
token_count = len(tokenizer.encode(long_text))
print(f"Token count: {token_count} tokens (BGE max: 512)")
print(f"Truncation will occur: {'YES ⚠️' if token_count > 512 else 'No ✓'}")

# Embed the full text and the second half separately
full_embed = embed_model.encode([long_text], normalize_embeddings=True)
second_half = long_text[len(long_text)//2:]
second_embed = embed_model.encode([second_half], normalize_embeddings=True)
query_embed = embed_model.encode(["What is the secret answer?"], normalize_embeddings=True)

sim_full = float(np.dot(full_embed[0], query_embed[0]))
sim_half = float(np.dot(second_embed[0], query_embed[0]))

print(f"\nQuery: 'What is the secret answer?'")
print(f"  Similarity with FULL text (truncated):  {sim_full:.4f}")
print(f"  Similarity with SECOND HALF only:       {sim_half:.4f}")
print(f"  Improvement from proper chunking:       {sim_half - sim_full:+.4f}")
print(f"\n💡 The second-half chunk scores HIGHER because the relevant sentence")
print(f"   isn't lost to truncation. This is why chunk_size < model_max_tokens matters!")

## When to Use Overlapping vs. Non-Overlapping Chunks

**Use overlap when:**
- Your text has important information at natural boundaries (paragraph edges, section transitions)
- Queries might target information that spans two adjacent paragraphs
- You're using fixed-size chunking (which ignores semantic boundaries)
- Storage cost is not a primary concern

**Skip overlap when:**
- You're already using semantic-boundary chunking (paragraphs, functions, sections)
- Your chunks are small enough that boundary losses are minimal
- Storage and index size are critical constraints (e.g., millions of documents)
- You need deterministic chunk-to-source mapping (overlap makes this ambiguous)

**Hybrid approach**: Use **non-overlapping semantic chunks** as the base, then add a
**sliding-window overlay** at a different granularity for boundary coverage. This gives you
the precision of semantic chunking with the boundary coverage of overlapping windows.

---
# Section 6: Synthesis — Making the Decision

## Decision Framework for Chunk Size Selection

Follow this flowchart when choosing chunk size for your RAG system:

```
1. What is your embedding model's max token limit?
   └─→ Your chunk size MUST be below this (e.g., 512 tokens for BGE-base)

2. What type of content are you indexing?
   ├─ Factoid Q&A / short answers → 256–512 chars
   ├─ Complex reasoning / multi-fact → 512–1024 chars
   ├─ Source code → function-level (variable size)
   └─ Legal / regulatory → section-level with hierarchy

3. What are your query patterns?
   ├─ Specific factual queries → smaller chunks (higher precision)
   └─ Broad / exploratory queries → larger chunks (higher recall)

4. Evaluate empirically:
   ├─ Create a test set of ~20-50 queries with known answers
   ├─ Test 3-5 chunk sizes spanning your candidate range
   └─ Measure precision, recall, and generation quality
```

In [ ]:
# Summary table of all our empirical results
print("=" * 85)
print("COMPREHENSIVE RESULTS SUMMARY")
print("=" * 85)
print(f"\n{'Chunk Size':>10} | {'# Chunks':>8} | {'P@3':>6} | {'R@3':>6} | {'F1@3':>6} | "
      f"{'Cos Sim':>7} | {'Embed(s)':>8} | {'Consec Sim':>10}")
print("-" * 85)

for cs in CHUNK_SIZES:
    r = results[cs]
    d = density.get(cs, {})
    f1 = 2 * r["precision@3"] * r["recall@3"] / (r["precision@3"] + r["recall@3"] + 1e-9)
    print(f"{cs:>10} | {r['n_chunks']:>8} | {r['precision@3']:>6.3f} | {r['recall@3']:>6.3f} | "
          f"{f1:>6.3f} | {r['avg_relevance_score']:>7.4f} | {r['embed_time']:>8.3f} | "
          f"{d.get('avg_consecutive_sim', 0):>10.4f}")

# Find the best chunk size by F1
best_cs = max(CHUNK_SIZES, key=lambda cs: (
    2 * results[cs]["precision@3"] * results[cs]["recall@3"] /
    (results[cs]["precision@3"] + results[cs]["recall@3"] + 1e-9)
))
best_f1 = 2 * results[best_cs]["precision@3"] * results[best_cs]["recall@3"] / (
    results[best_cs]["precision@3"] + results[best_cs]["recall@3"] + 1e-9)
print(f"\n🏆 Best chunk size by F1@3: {best_cs} characters (F1 = {best_f1:.3f})")
print(f"   Precision@3 = {results[best_cs]['precision@3']:.3f}, "
      f"Recall@3 = {results[best_cs]['recall@3']:.3f}")

## Rule-of-Thumb Recommendations

Based on our experiments and broader RAG literature:

| Scenario | Chunk Size | Overlap | Strategy |
|---|---|---|---|
| **General Q&A** | 512 chars (~128 tokens) | 15% | Sentence-boundary |
| **Technical docs** | 1024 chars (~256 tokens) | 20% | Paragraph-boundary |
| **Legal / compliance** | 1500 chars (~375 tokens) | 10% | Section-boundary |
| **Source code** | Variable | 0% | Function-level |
| **Chat / conversation** | Variable | 1-2 turns | Turn-boundary |

**Universal rules:**
1. **Never exceed your embedding model's max tokens** — this is a hard constraint
2. **Start with 512 chars + sentence boundaries** — it's a strong baseline
3. **Always evaluate empirically** — domain-specific data always beats rules of thumb
4. **Use overlap sparingly** (10-20%) — diminishing returns are real
5. **Re-evaluate when you change content types** — what works for docs won't work for code

## Common Mistakes and How to Avoid Them

**Mistake 1: Ignoring embedding model limits**
> Using 4096-char chunks with a 512-token embedding model. Half your chunk is invisible.
> **Fix**: Always check `embed_model.max_seq_length` and stay well below it.

**Mistake 2: One-size-fits-all chunking**
> Using the same chunk size for code, prose, and tables.
> **Fix**: Use content-type detection and apply appropriate strategies per type.

**Mistake 3: No overlap on fixed-size chunks**
> Splitting every 500 characters with no overlap — boundary sentences get split.
> **Fix**: Add 10-20% overlap, or switch to sentence-boundary chunking.

**Mistake 4: Tuning chunk size without evaluation**
> Picking 512 tokens "because everyone uses it" without measuring on your data.
> **Fix**: Build an evaluation set and measure. It takes an hour and saves weeks of debugging.

**Mistake 5: Optimizing chunk size in isolation**
> Tuning chunks without considering the downstream LLM context window and token budget.
> **Fix**: Co-optimize chunk size, top-k, and prompt template together.

---

**Key takeaway**: Chunk size is not a hyperparameter you set once and forget. It's a
**design decision** that interacts with your embedding model, retrieval strategy, content type,
and query patterns. The empirical framework in this notebook gives you the tools to make
that decision rigorously for any domain.